# Hyperparameter Optimization (Phase 4)

Two-stage tuning policy for the Top-3 models (from `results/top3_models.csv`), on every dataset and every shared output target:

1. **Random Search** — broad exploration over the registry's search space (`configs/config.yaml: tuning.random_search_iterations`).
2. **Bayesian Optimization (Optuna, TPE sampler)** — focused refinement (`tuning.optuna_trials`).

Grid Search is intentionally not implemented (forbidden for this small-data project). Best CV RMSE and best parameters from both stages are saved to `results/tuning/top3_tuning_results.csv`, and the last cell shows the complete results.

Setup: repo imports, pandas display options, load the Top-3 models and all datasets.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if (Path.cwd() / '..' / 'src').resolve().exists() else Path.cwd()
SCRIPTS_DIR = REPO_ROOT / 'scripts'
for p in (REPO_ROOT, SCRIPTS_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

from _pipeline_common import read_top3_models
from src.config import get_base_seed, get_path, load_config
from src.data.loader import load_all_datasets
from src.tuning.hyperparameter import optuna_tune, random_search
from src.utils.io import save_table

cfg = load_config()
seed = get_base_seed()
top3 = read_top3_models()
bundles = load_all_datasets(discrete=False)
print('Top-3 models to tune:', top3)
print('Random search iterations:', cfg['tuning']['random_search_iterations'])
print('Optuna trials:', cfg['tuning']['optuna_trials'])

## Tuning on Dataset_0136

Random Search followed by Optuna Bayesian optimization, for every shared target and every Top-3 model.

In [ ]:
rows_0136 = []
bundle = bundles['Dataset_0136']
shared_targets = [c for c in cfg['shared_outputs'] if c in bundle.Y.columns]
for target in shared_targets:
    y = bundle.Y[target]
    for model_name in top3:
        rs = random_search(model_name, bundle.X, y, seed=seed)
        study = optuna_tune(model_name, bundle.X, y, seed=seed)
        rows_0136.append({
            'dataset': 'Dataset_0136',
            'target': target,
            'model': model_name,
            'random_search_best_rmse': -rs.best_score_,
            'random_search_best_params': str(rs.best_params_),
            'optuna_best_rmse': -study.best_value,
            'optuna_best_params': str(study.best_params),
        })
        print(f'[tuning] Dataset_0136 / {target} / {model_name} done')

tuning_0136 = pd.DataFrame(rows_0136)
tuning_0136

## Tuning on Dataset_0172

Random Search followed by Optuna Bayesian optimization, for every shared target and every Top-3 model.

In [ ]:
rows_0172 = []
bundle = bundles['Dataset_0172']
shared_targets = [c for c in cfg['shared_outputs'] if c in bundle.Y.columns]
for target in shared_targets:
    y = bundle.Y[target]
    for model_name in top3:
        rs = random_search(model_name, bundle.X, y, seed=seed)
        study = optuna_tune(model_name, bundle.X, y, seed=seed)
        rows_0172.append({
            'dataset': 'Dataset_0172',
            'target': target,
            'model': model_name,
            'random_search_best_rmse': -rs.best_score_,
            'random_search_best_params': str(rs.best_params_),
            'optuna_best_rmse': -study.best_value,
            'optuna_best_params': str(study.best_params),
        })
        print(f'[tuning] Dataset_0172 / {target} / {model_name} done')

tuning_0172 = pd.DataFrame(rows_0172)
tuning_0172

## Tuning on Dataset_3772

Random Search followed by Optuna Bayesian optimization, for every shared target and every Top-3 model.

In [ ]:
rows_3772 = []
bundle = bundles['Dataset_3772']
shared_targets = [c for c in cfg['shared_outputs'] if c in bundle.Y.columns]
for target in shared_targets:
    y = bundle.Y[target]
    for model_name in top3:
        rs = random_search(model_name, bundle.X, y, seed=seed)
        study = optuna_tune(model_name, bundle.X, y, seed=seed)
        rows_3772.append({
            'dataset': 'Dataset_3772',
            'target': target,
            'model': model_name,
            'random_search_best_rmse': -rs.best_score_,
            'random_search_best_params': str(rs.best_params_),
            'optuna_best_rmse': -study.best_value,
            'optuna_best_params': str(study.best_params),
        })
        print(f'[tuning] Dataset_3772 / {target} / {model_name} done')

tuning_3772 = pd.DataFrame(rows_3772)
tuning_3772

## Final Summary — Complete Hyperparameter Tuning Results (All Datasets)

Random Search vs Optuna best RMSE, side by side, saved to `results/tuning/top3_tuning_results.csv`.

In [ ]:
tuning_all = pd.concat([tuning_0136, tuning_0172, tuning_3772], ignore_index=True)
save_table(tuning_all, get_path('results_dir') / 'tuning' / 'top3_tuning_results.csv')

print('=' * 90)
print('HYPERPARAMETER TUNING RESULTS — Random Search vs Optuna (all datasets)')
print('=' * 90)
display(tuning_all)

improvement = tuning_all.copy()
improvement['optuna_improves_over_random'] = (
    improvement['optuna_best_rmse'] < improvement['random_search_best_rmse']
)
print()
print('Optuna improved on Random Search in',
      f"{improvement['optuna_improves_over_random'].sum()} / {len(improvement)}",
      'model/target/dataset combinations')

print()
print('Tuning results saved to:', (get_path('results_dir') / 'tuning').resolve())